# Artificial Intelligence Nanodegree
## Machine Translation Project

## Introduction
In this notebook, i will build a deep neural network that functions as part of an end-to-end machine translation pipeline. My completed pipeline will accept English text as input and return the French translation.

- **Preprocess** - I'll convert text to sequence of integers.
- **Models** Create models which accepts a sequence of integers as input and returns a probability distribution over possible translations. After learning about the basic types of neural networks that are often used for machine translation, i will engage in your own investigations, to design my own model!
- **Prediction** Run the model on English text.

In [1]:
%load_ext autoreload
%aimport helper, project_tests
%autoreload 1

2025-12-15 12:33:03.401102: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-15 12:33:03.401176: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-15 12:33:03.401280: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-15 12:33:03.414610: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Should return keras==2.14.0 and tf-keras==2.14.1
!pip freeze | grep keras

keras==2.14.0
tf-keras==2.14.1


In [3]:
# Should return tensorflow==2.14.0
!pip freeze | grep tensorflow

tensorflow==2.14.0
tensorflow-datasets==4.9.4
tensorflow-estimator==2.14.0
tensorflow-hub==0.16.1
tensorflow-io-gcs-filesystem==0.34.0
tensorflow-metadata==1.15.0


In [4]:
import collections

import helper
import numpy as np
import project_tests as tests

from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Model
from tensorflow.keras.layers import GRU, Input, Dense, TimeDistributed, Activation, RepeatVector, Bidirectional
from keras.layers import Embedding
from keras.optimizers import Adam
from keras.losses import sparse_categorical_crossentropy

## Dataset
We begin by investigating the dataset that will be used to train and evaluate your pipeline.  The most common datasets used for machine translation are from [WMT](http://www.statmt.org/).  However, that will take a long time to train a neural network on.  We'll be using a dataset we created for this project that contains a small vocabulary.  i'll be able to train your model in a reasonable time with this dataset.
### Load Data
The data is located in `data/small_vocab_en` and `data/small_vocab_fr`. The `small_vocab_en` file contains English sentences with their French translations in the `small_vocab_fr` file. Load the English and French data from these files from running the cell below.

In [5]:
# Load English data
english_sentences = helper.load_data('data/small_vocab_en')
# Load French data
french_sentences = helper.load_data('data/small_vocab_fr')

print('Dataset Loaded')

Dataset Loaded


### Files
Each line in `small_vocab_en` contains an English sentence with the respective translation in each line of `small_vocab_fr`.  View the first two lines from each file.

In [6]:
for sample_i in range(2):
    print('small_vocab_en Line {}:  {}'.format(sample_i + 1, english_sentences[sample_i]))
    print('small_vocab_fr Line {}:  {}'.format(sample_i + 1, french_sentences[sample_i]))

small_vocab_en Line 1:  new jersey is sometimes quiet during autumn , and it is snowy in april .
small_vocab_fr Line 1:  new jersey est parfois calme pendant l' automne , et il est neigeux en avril .
small_vocab_en Line 2:  the united states is usually chilly during july , and it is usually freezing in november .
small_vocab_fr Line 2:  les états-unis est généralement froid en juillet , et il gèle habituellement en novembre .


From looking at the sentences, i can see they have been preprocessed already.  The puncuations have been delimited using spaces. All the text have been converted to lowercase.  This should save some time, but the text requires more preprocessing.
### Vocabulary
The complexity of the problem is determined by the complexity of the vocabulary.  A more complex vocabulary is a more complex problem.  Let's look at the complexity of the dataset we'll be working with.

In [7]:
english_words_counter = collections.Counter([word for sentence in english_sentences for word in sentence.split()])
french_words_counter = collections.Counter([word for sentence in french_sentences for word in sentence.split()])

print('{} English words.'.format(len([word for sentence in english_sentences for word in sentence.split()])))
print('{} unique English words.'.format(len(english_words_counter)))
print('10 Most common words in the English dataset:')
print('"' + '" "'.join(list(zip(*english_words_counter.most_common(10)))[0]) + '"')
print()
print('{} French words.'.format(len([word for sentence in french_sentences for word in sentence.split()])))
print('{} unique French words.'.format(len(french_words_counter)))
print('10 Most common words in the French dataset:')
print('"' + '" "'.join(list(zip(*french_words_counter.most_common(10)))[0]) + '"')

1823250 English words.
227 unique English words.
10 Most common words in the English dataset:
"is" "," "." "in" "it" "during" "the" "but" "and" "sometimes"

1961295 French words.
355 unique French words.
10 Most common words in the French dataset:
"est" "." "," "en" "il" "les" "mais" "et" "la" "parfois"


For comparison, _Alice's Adventures in Wonderland_ contains 2,766 unique words of a total of 15,500 words.
## Preprocess
For this project, i won't use text data as input to your model. Instead, i'll convert the text into sequences of integers using the following preprocess methods:
1. Tokenize the words into ids
2. Add padding to make all the sequences the same length.

Time to start preprocessing the data...
### Tokenize (IMPLEMENTATION)
For a neural network to predict on text data, it first has to be turned into data it can understand. Text data like "dog" is a sequence of ASCII character encodings.  Since a neural network is a series of multiplication and addition operations, the input data needs to be number(s).

We can turn each character into a number or each word into a number.  These are called character and word ids, respectively.  Character ids are used for character level models that generate text predictions for each character.  A word level model uses word ids that generate text predictions for each word.  Word level models tend to learn better, since they are lower in complexity, so we'll use those.

Turn each sentence into a sequence of words ids using Keras's [`Tokenizer`](https://keras.io/preprocessing/text/#tokenizer) function. Use this function to tokenize `english_sentences` and `french_sentences` in the cell below.

Running the cell will run `tokenize` on sample data and show output for debugging.

In [8]:
def tokenize(x):
    """
    Tokenize x
    :param x: List of sentences/strings to be tokenized
    :return: Tuple of (tokenized x data, tokenizer used to tokenize x)
    """
    # TODO: Implement
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(x)
    tokenized_x = tokenizer.texts_to_sequences(x)
    return tokenized_x, tokenizer

tests.test_tokenize(tokenize)

# Tokenize Example output
text_sentences = [
    'The quick brown fox jumps over the lazy dog .',
    'By Jove , my quick study of lexicography won a prize .',
    'This is a short sentence .']
text_tokenized, text_tokenizer = tokenize(text_sentences)
print(text_tokenizer.word_index)
print()
for sample_i, (sent, token_sent) in enumerate(zip(text_sentences, text_tokenized)):
    print('Sequence {} in x'.format(sample_i + 1))
    print('  Input:  {}'.format(sent))
    print('  Output: {}'.format(token_sent))


{'the': 1, 'quick': 2, 'a': 3, 'brown': 4, 'fox': 5, 'jumps': 6, 'over': 7, 'lazy': 8, 'dog': 9, 'by': 10, 'jove': 11, 'my': 12, 'study': 13, 'of': 14, 'lexicography': 15, 'won': 16, 'prize': 17, 'this': 18, 'is': 19, 'short': 20, 'sentence': 21}

Sequence 1 in x
  Input:  The quick brown fox jumps over the lazy dog .
  Output: [1, 2, 4, 5, 6, 7, 1, 8, 9]
Sequence 2 in x
  Input:  By Jove , my quick study of lexicography won a prize .
  Output: [10, 11, 12, 2, 13, 14, 15, 16, 3, 17]
Sequence 3 in x
  Input:  This is a short sentence .
  Output: [18, 19, 3, 20, 21]


### Padding (IMPLEMENTATION)
When batching the sequence of word ids together, each sequence needs to be the same length.  Since sentences are dynamic in length, we can add padding to the end of the sequences to make them the same length.

I will make sure all the English sequences have the same length and all the French sequences have the same length by adding padding to the **end** of each sequence using Keras's [`pad_sequences`](https://keras.io/preprocessing/sequence/#pad_sequences) function.

In [9]:
def pad(x, length=None):
    """
    Pad x
    :param x: List of sequences.
    :param length: Length to pad the sequence to.  If None, use length of longest sequence in x.
    :return: Padded numpy array of sequences
    """
    # TODO: Implement
    if length is None:
        length = max(len(seq) for seq in x)
    return pad_sequences(x, maxlen=length, padding='post')

tests.test_pad(pad)

# Pad Tokenized output
test_pad = pad(text_tokenized)
for sample_i, (token_sent, pad_sent) in enumerate(zip(text_tokenized, test_pad)):
    print('Sequence {} in x'.format(sample_i + 1))
    print('  Input:  {}'.format(np.array(token_sent)))
    print('  Output: {}'.format(pad_sent))


Sequence 1 in x
  Input:  [1 2 4 5 6 7 1 8 9]
  Output: [1 2 4 5 6 7 1 8 9 0]
Sequence 2 in x
  Input:  [10 11 12  2 13 14 15 16  3 17]
  Output: [10 11 12  2 13 14 15 16  3 17]
Sequence 3 in x
  Input:  [18 19  3 20 21]
  Output: [18 19  3 20 21  0  0  0  0  0]


### Preprocess Pipeline
My focus for this project is to build neural network architecture, so i won't create a preprocess pipeline.  Instead, i have the implementation of the `preprocess` function.

In [10]:
def preprocess(x, y):
    """
    Preprocess x and y
    :param x: Feature List of sentences
    :param y: Label List of sentences
    :return: Tuple of (Preprocessed x, Preprocessed y, x tokenizer, y tokenizer)
    """
    preprocess_x, x_tk = tokenize(x)
    preprocess_y, y_tk = tokenize(y)

    preprocess_x = pad(preprocess_x)
    preprocess_y = pad(preprocess_y)

    # Keras's sparse_categorical_crossentropy function requires the labels to be in 3 dimensions
    preprocess_y = preprocess_y.reshape(*preprocess_y.shape, 1)

    return preprocess_x, preprocess_y, x_tk, y_tk

preproc_english_sentences, preproc_french_sentences, english_tokenizer, french_tokenizer =\
    preprocess(english_sentences, french_sentences)
    
max_english_sequence_length = preproc_english_sentences.shape[1]
max_french_sequence_length = preproc_french_sentences.shape[1]
english_vocab_size = len(english_tokenizer.word_index)
french_vocab_size = len(french_tokenizer.word_index)

print('Data Preprocessed')
print("Max English sentence length:", max_english_sequence_length)
print("Max French sentence length:", max_french_sequence_length)
print("English vocabulary size:", english_vocab_size)
print("French vocabulary size:", french_vocab_size)

Data Preprocessed
Max English sentence length: 15
Max French sentence length: 21
English vocabulary size: 199
French vocabulary size: 344


## Models
In this section, i will experiment with various neural network architectures.
I will begin by training four relatively simple architectures.
- Model 1 is a simple RNN
- Model 2 is a RNN with Embedding
- Model 3 is a Bidirectional RNN
- Model 4 is an Encoder-Decoder RNN

After experimenting with the four simple architectures, i will construct a deeper architecture that is designed to outperform all four models.
### Ids Back to Text
The neural network will be translating the input to words ids, which isn't the final form we want.  We want the French translation.  The function `logits_to_text` will bridge the gab between the logits from the neural network to the French translation.  i'll be using this function to better understand the output of the neural network.

In [11]:
def logits_to_text(logits, tokenizer):
    """
    Turn logits from a neural network into text using the tokenizer
    :param logits: Logits from a neural network
    :param tokenizer: Keras Tokenizer fit on the labels
    :return: String that represents the text of the logits
    """
    index_to_words = {id: word for word, id in tokenizer.word_index.items()}
    index_to_words[0] = '<PAD>'

    return ' '.join([index_to_words[prediction] for prediction in np.argmax(logits, 1)])

print('`logits_to_text` function loaded.')

`logits_to_text` function loaded.


### Model 1: RNN (IMPLEMENTATION)
![RNN](images/rnn.png)
A basic RNN model is a good baseline for sequence data.  In this model, i'll build a RNN that translates English to French.

In [12]:
def simple_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Build and train a basic RNN on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    # TODO: Build the layers
    input_layer = Input(shape=input_shape[1:])
    rnn = GRU(64, return_sequences=True)(input_layer)
    logits = TimeDistributed(Dense(french_vocab_size, activation='softmax'))(rnn)

    model = Model(input_layer, logits)
    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(),
                  metrics=['accuracy'])
    return model


tests.test_simple_model(simple_model)

# Reshaping the input to work with a basic RNN
tmp_x = pad(preproc_english_sentences, max_french_sequence_length)
tmp_x = tmp_x.reshape((-1, preproc_french_sentences.shape[-2], 1))

# Train the neural network
simple_rnn_model = simple_model(
    tmp_x.shape,
    max_french_sequence_length,
    english_vocab_size,
    french_vocab_size)

simple_rnn_model.fit(
    tmp_x,
    preproc_french_sentences,
    batch_size=1024,
    epochs=10,
    validation_split=0.2
)

# Print prediction(s)
print(logits_to_text(simple_rnn_model.predict(tmp_x[:1])[0], french_tokenizer))


2025-12-15 12:33:18.439967: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-12-15 12:33:18.451106: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-12-15 12:33:18.453383: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

Epoch 1/10


2025-12-15 12:33:21.753414: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8600
2025-12-15 12:33:23.820833: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x79aff4e8a550 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-12-15 12:33:23.820892: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
2025-12-15 12:33:23.826754: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-12-15 12:33:23.947244: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


108/108 [==============================] - 6s 19ms/step - loss: 3.5124 - accuracy: 0.4045 - val_loss: nan - val_accuracy: 0.4601
Epoch 2/10
108/108 [==============================] - 1s 14ms/step - loss: 2.4556 - accuracy: 0.4700 - val_loss: nan - val_accuracy: 0.4886
Epoch 3/10
108/108 [==============================] - 1s 14ms/step - loss: 2.1600 - accuracy: 0.5168 - val_loss: nan - val_accuracy: 0.5370
Epoch 4/10
108/108 [==============================] - 1s 14ms/step - loss: 1.9127 - accuracy: 0.5546 - val_loss: nan - val_accuracy: 0.5691
Epoch 5/10
108/108 [==============================] - 1s 14ms/step - loss: 1.7575 - accuracy: 0.5730 - val_loss: nan - val_accuracy: 0.5765
Epoch 6/10
108/108 [==============================] - 1s 14ms/step - loss: 1.6652 - accuracy: 0.5785 - val_loss: nan - val_accuracy: 0.5819
Epoch 7/10
108/108 [==============================] - 1s 14ms/step - loss: 1.6002 - accuracy: 0.5818 - val_loss: nan - val_accuracy: 0.5836
Epoch 8/10
108/108 [===========

### Model 2: Embedding (IMPLEMENTATION)
![RNN](images/embedding.png)
I have turned the words into ids, but there's a better representation of a word.  This is called word embeddings.  An embedding is a vector representation of the word that is close to similar words in n-dimensional space, where the n represents the size of the embedding vectors.

In this model, i'll create a RNN model using embedding.

In [13]:
def embed_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Build and train a RNN model using word embedding on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    # TODO: Implement
    input_layer = Input(shape=input_shape[1:])
    embedding = Embedding(english_vocab_size + 1, 64)(input_layer)
    rnn = GRU(64, return_sequences=True)(embedding)
    logits = TimeDistributed(Dense(french_vocab_size, activation='softmax'))(rnn)

    model = Model(input_layer, logits)
    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(),
                  metrics=['accuracy'])
    return model


tests.test_embed_model(embed_model)


# TODO: Reshape the input
tmp_x = pad(preproc_english_sentences, max_french_sequence_length)


# TODO: Train the neural network
embed_rnn_model = embed_model(
    tmp_x.shape,
    max_french_sequence_length,
    english_vocab_size,
    french_vocab_size)

embed_rnn_model.fit(
    tmp_x,
    preproc_french_sentences,
    batch_size=1024,
    epochs=10,
    validation_split=0.2
)


# TODO: Print prediction(s)
print(logits_to_text(embed_rnn_model.predict(tmp_x[:1])[0], french_tokenizer))


Epoch 1/10
108/108 [==============================] - 8s 50ms/step - loss: 3.7370 - accuracy: 0.4012 - val_loss: nan - val_accuracy: 0.4095
Epoch 2/10
108/108 [==============================] - 2s 20ms/step - loss: 2.6301 - accuracy: 0.4634 - val_loss: nan - val_accuracy: 0.5119
Epoch 3/10
108/108 [==============================] - 2s 17ms/step - loss: 1.9701 - accuracy: 0.5595 - val_loss: nan - val_accuracy: 0.6094
Epoch 4/10
108/108 [==============================] - 2s 17ms/step - loss: 1.4603 - accuracy: 0.6455 - val_loss: nan - val_accuracy: 0.6943
Epoch 5/10
108/108 [==============================] - 2s 15ms/step - loss: 1.1228 - accuracy: 0.7268 - val_loss: nan - val_accuracy: 0.7515
Epoch 6/10
108/108 [==============================] - 2s 15ms/step - loss: 0.9182 - accuracy: 0.7648 - val_loss: nan - val_accuracy: 0.7801
Epoch 7/10
108/108 [==============================] - 2s 16ms/step - loss: 0.7853 - accuracy: 0.7905 - val_loss: nan - val_accuracy: 0.8014
Epoch 8/10
108/108 [

### Model 3: Bidirectional RNNs (IMPLEMENTATION)
![RNN](images/bidirectional.png)
One restriction of a RNN is that it can't see the future input, only the past.  This is where bidirectional recurrent neural networks come in.  They are able to see the future data.

In [14]:
def bd_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Build and train a bidirectional RNN model on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    # TODO: Implement
    input_layer = Input(shape=input_shape[1:])
    bd_rnn = Bidirectional(GRU(64, return_sequences=True))(input_layer)
    logits = TimeDistributed(Dense(french_vocab_size, activation='softmax'))(bd_rnn)

    model = Model(input_layer, logits)
    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(),
                  metrics=['accuracy'])
    return model


tests.test_bd_model(bd_model)


# TODO: Train and Print prediction(s)
tmp_x = pad(preproc_english_sentences, max_french_sequence_length)
tmp_x = tmp_x.reshape((-1, preproc_french_sentences.shape[-2], 1))

bd_rnn_model = bd_model(
    tmp_x.shape,
    max_french_sequence_length,
    english_vocab_size,
    french_vocab_size)

bd_rnn_model.fit(
    tmp_x,
    preproc_french_sentences,
    batch_size=1024,
    epochs=10,
    validation_split=0.2
)

print(logits_to_text(bd_rnn_model.predict(tmp_x[:1])[0], french_tokenizer))


Epoch 1/10
108/108 [==============================] - 6s 25ms/step - loss: 3.2522 - accuracy: 0.4643 - val_loss: nan - val_accuracy: 0.5028
Epoch 2/10
108/108 [==============================] - 2s 17ms/step - loss: 1.9912 - accuracy: 0.5440 - val_loss: nan - val_accuracy: 0.5720
Epoch 3/10
108/108 [==============================] - 2s 17ms/step - loss: 1.6293 - accuracy: 0.5884 - val_loss: nan - val_accuracy: 0.6043
Epoch 4/10
108/108 [==============================] - 2s 17ms/step - loss: 1.4718 - accuracy: 0.6163 - val_loss: nan - val_accuracy: 0.6239
Epoch 5/10
108/108 [==============================] - 2s 18ms/step - loss: 1.3830 - accuracy: 0.6296 - val_loss: nan - val_accuracy: 0.6355
Epoch 6/10
108/108 [==============================] - 2s 17ms/step - loss: 1.3157 - accuracy: 0.6397 - val_loss: nan - val_accuracy: 0.6454
Epoch 7/10
108/108 [==============================] - 2s 17ms/step - loss: 1.2598 - accuracy: 0.6486 - val_loss: nan - val_accuracy: 0.6544
Epoch 8/10
108/108 [

### Model 4: Encoder-Decoder (IMPLEMENTATION)
Time to look at encoder-decoder models.  This model is made up of an encoder and decoder. The encoder creates a matrix representation of the sentence.  The decoder takes this matrix as input and predicts the translation as output.

encoder-decoder model in the cell below.

In [15]:
def encdec_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Build and train an encoder-decoder model on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    # TODO: Implement
    input_layer = Input(shape=input_shape[1:])
    
    # Encoder
    encoder = GRU(64, return_sequences=False)(input_layer)
    
    # Decoder
    decoder_input = RepeatVector(output_sequence_length)(encoder)
    decoder = GRU(64, return_sequences=True)(decoder_input)
    
    logits = TimeDistributed(Dense(french_vocab_size, activation='softmax'))(decoder)

    model = Model(input_layer, logits)
    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(),
                  metrics=['accuracy'])
    return model


tests.test_encdec_model(encdec_model)


# TODO: Train and Print prediction(s)
tmp_x = pad(preproc_english_sentences, max_french_sequence_length)
tmp_x = tmp_x.reshape((-1, preproc_french_sentences.shape[-2], 1))

encdec_rnn_model = encdec_model(
    tmp_x.shape,
    max_french_sequence_length,
    english_vocab_size,
    french_vocab_size)

encdec_rnn_model.fit(
    tmp_x,
    preproc_french_sentences,
    batch_size=1024,
    epochs=10,
    validation_split=0.2
)

print(logits_to_text(encdec_rnn_model.predict(tmp_x[:1])[0], french_tokenizer))


Epoch 1/10
108/108 [==============================] - 7s 24ms/step - loss: 3.5718 - accuracy: 0.4079 - val_loss: nan - val_accuracy: 0.4443
Epoch 2/10
108/108 [==============================] - 2s 17ms/step - loss: 2.5635 - accuracy: 0.4729 - val_loss: nan - val_accuracy: 0.4926
Epoch 3/10
108/108 [==============================] - 2s 17ms/step - loss: 2.3354 - accuracy: 0.4956 - val_loss: nan - val_accuracy: 0.4974
Epoch 4/10
108/108 [==============================] - 2s 17ms/step - loss: 2.2249 - accuracy: 0.5004 - val_loss: nan - val_accuracy: 0.5022
Epoch 5/10
108/108 [==============================] - 2s 17ms/step - loss: 2.0866 - accuracy: 0.5112 - val_loss: nan - val_accuracy: 0.5210
Epoch 6/10
108/108 [==============================] - 2s 17ms/step - loss: 1.9102 - accuracy: 0.5366 - val_loss: nan - val_accuracy: 0.5481
Epoch 7/10
108/108 [==============================] - 2s 17ms/step - loss: 1.7992 - accuracy: 0.5506 - val_loss: nan - val_accuracy: 0.5538
Epoch 8/10
108/108 [

### Model 5: Custom (IMPLEMENTATION)
Use everything learned from the previous models to create a model that incorporates embedding and a bidirectional rnn into one model.

In [23]:
from keras.layers import Dot, Activation, Concatenate

def model_final(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Final Model: Encoder–Decoder with Attention
    """
    # Encoder
    encoder_inputs = Input(shape=input_shape[1:])
    encoder_embedding = Embedding(english_vocab_size + 1, 128)(encoder_inputs)

    # Bidirectional encoder: 64 * 2 = 128
    encoder_outputs = Bidirectional(
        GRU(64, return_sequences=True)
    )(encoder_embedding)

    # Decoder (matches encoder feature size = 128)
    decoder_inputs = RepeatVector(output_sequence_length)(encoder_outputs[:, -1, :])
    decoder_outputs = GRU(128, return_sequences=True)(decoder_inputs)

    # Attention
    attention_scores = Dot(axes=[2, 2])([decoder_outputs, encoder_outputs])
    attention_weights = Activation('softmax')(attention_scores)
    context_vector = Dot(axes=[2, 1])([attention_weights, encoder_outputs])

    # Combine context and decoder outputs
    decoder_combined = Concatenate()([context_vector, decoder_outputs])

    # Output
    logits = TimeDistributed(Dense(french_vocab_size, activation='softmax'))(decoder_combined)

    model = Model(encoder_inputs, logits)
    model.compile(
        loss=sparse_categorical_crossentropy,
        optimizer=Adam(),
        metrics=['accuracy']
    )
    return model



tests.test_model_final(model_final)

print('Final Model Loaded')

# Train the final model
tmp_x = pad(preproc_english_sentences, max_french_sequence_length)

final_model = model_final(
    tmp_x.shape,
    max_french_sequence_length,
    english_vocab_size,
    french_vocab_size
)

final_model.fit(
    tmp_x,
    preproc_french_sentences,
    batch_size=1024,
    epochs=40,
    validation_split=0.2
)


Final Model Loaded
Epoch 1/40


2025-12-15 13:04:54.182035: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:693] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla T4" frequency: 1590 num_cores: 40 environment { key: "architecture" value: "7.5" } environment { key: "cuda" value: "11080" } environment { key: "cudnn" value: "8600" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 4194304 shared_memory_size_per_multiprocessor: 65536 memory_size: 14426112000 bandwidth: 320064000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


108/108 [==============================] - ETA: 0s - loss: 3.1584 - accuracy: 0.4265

2025-12-15 13:05:00.839658: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:693] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla T4" frequency: 1590 num_cores: 40 environment { key: "architecture" value: "7.5" } environment { key: "cuda" value: "11080" } environment { key: "cudnn" value: "8600" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 4194304 shared_memory_size_per_multiprocessor: 65536 memory_size: 14426112000 bandwidth: 320064000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


108/108 [==============================] - 12s 57ms/step - loss: 3.1584 - accuracy: 0.4265 - val_loss: nan - val_accuracy: 0.4808
Epoch 2/40
108/108 [==============================] - 4s 37ms/step - loss: 1.9556 - accuracy: 0.5062 - val_loss: nan - val_accuracy: 0.5440
Epoch 3/40
108/108 [==============================] - 4s 37ms/step - loss: 1.6651 - accuracy: 0.5715 - val_loss: nan - val_accuracy: 0.6012
Epoch 4/40
108/108 [==============================] - 4s 35ms/step - loss: 1.4151 - accuracy: 0.6243 - val_loss: nan - val_accuracy: 0.6511
Epoch 5/40
108/108 [==============================] - 4s 37ms/step - loss: 1.2166 - accuracy: 0.6717 - val_loss: nan - val_accuracy: 0.6900
Epoch 6/40
108/108 [==============================] - 4s 34ms/step - loss: 1.0921 - accuracy: 0.7007 - val_loss: nan - val_accuracy: 0.7129
Epoch 7/40
108/108 [==============================] - 4s 35ms/step - loss: 1.0071 - accuracy: 0.7213 - val_loss: nan - val_accuracy: 0.7336
Epoch 8/40
108/108 [==========

## Prediction (IMPLEMENTATION)

In [24]:
def final_predictions(x, y, x_tk, y_tk):
    """
    Gets predictions using the final model
    :param x: Preprocessed English data
    :param y: Preprocessed French data
    :param x_tk: English tokenizer
    :param y_tk: French tokenizer
    """
    # TODO: Train neural network using model_final
    model = model_final(
        x.shape,
        y.shape[1],
        len(x_tk.word_index),
        len(y_tk.word_index)
    )
    
    model.fit(
        x,
        y,
        batch_size=1024,
        epochs=40,
        validation_split=0.2
    )

    ## DON'T EDIT ANYTHING BELOW THIS LINE
    y_id_to_word = {value: key for key, value in y_tk.word_index.items()}
    y_id_to_word[0] = '<PAD>'

    sentence = 'he saw a old yellow truck'
    sentence = [x_tk.word_index[word] for word in sentence.split()]
    sentence = pad_sequences([sentence], maxlen=x.shape[-1], padding='post')
    sentences = np.array([sentence[0], x[0]])
    predictions = model.predict(sentences, len(sentences))

    print('Sample 1:')
    print(' '.join([y_id_to_word[np.argmax(x)] for x in predictions[0]]))
    print('Il a vu un vieux camion jaune')
    print('Sample 2:')
    print(' '.join([y_id_to_word[np.argmax(x)] for x in predictions[1]]))
    print(' '.join([y_id_to_word[np.max(x)] for x in y[0]]))


final_predictions(preproc_english_sentences, preproc_french_sentences, english_tokenizer, french_tokenizer)


Epoch 1/40


2025-12-15 13:07:30.273507: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:693] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla T4" frequency: 1590 num_cores: 40 environment { key: "architecture" value: "7.5" } environment { key: "cuda" value: "11080" } environment { key: "cudnn" value: "8600" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 4194304 shared_memory_size_per_multiprocessor: 65536 memory_size: 14426112000 bandwidth: 320064000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


107/108 [============================>.] - ETA: 0s - loss: 3.1632 - accuracy: 0.4204

2025-12-15 13:07:36.761126: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:693] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla T4" frequency: 1590 num_cores: 40 environment { key: "architecture" value: "7.5" } environment { key: "cuda" value: "11080" } environment { key: "cudnn" value: "8600" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 4194304 shared_memory_size_per_multiprocessor: 65536 memory_size: 14426112000 bandwidth: 320064000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


108/108 [==============================] - 12s 55ms/step - loss: 3.1568 - accuracy: 0.4207 - val_loss: nan - val_accuracy: 0.4798
Epoch 2/40
108/108 [==============================] - 3s 32ms/step - loss: 1.9406 - accuracy: 0.5138 - val_loss: nan - val_accuracy: 0.5424
Epoch 3/40
108/108 [==============================] - 4s 33ms/step - loss: 1.6773 - accuracy: 0.5686 - val_loss: nan - val_accuracy: 0.5977
Epoch 4/40
108/108 [==============================] - 4s 33ms/step - loss: 1.4726 - accuracy: 0.6133 - val_loss: nan - val_accuracy: 0.6332
Epoch 5/40
108/108 [==============================] - 3s 32ms/step - loss: 1.3443 - accuracy: 0.6411 - val_loss: nan - val_accuracy: 0.6503
Epoch 6/40
108/108 [==============================] - 3s 32ms/step - loss: 1.2352 - accuracy: 0.6682 - val_loss: nan - val_accuracy: 0.6827
Epoch 7/40
108/108 [==============================] - 3s 32ms/step - loss: 1.1377 - accuracy: 0.6912 - val_loss: nan - val_accuracy: 0.7014
Epoch 8/40
108/108 [==========

2025-12-15 13:09:49.926375: W tensorflow/core/grappler/costs/op_level_cost_estimator.cc:693] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "GPU" vendor: "NVIDIA" model: "Tesla T4" frequency: 1590 num_cores: 40 environment { key: "architecture" value: "7.5" } environment { key: "cuda" value: "11080" } environment { key: "cudnn" value: "8600" } num_registers: 65536 l1_cache_size: 24576 l2_cache_size: 4194304 shared_memory_size_per_multiprocessor: 65536 memory_size: 14426112000 bandwidth: 320064000 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


### Generate the html

**Save the notebook before running the next cell to generate the HTML output.**

In [25]:
!jupyter nbconvert --to html machine_translation.ipynb

[NbConvertApp] Converting notebook machine_translation.ipynb to html
[NbConvertApp] Writing 396971 bytes to machine_translation.html


## Project Overview and Design Rationale

This project implements an end-to-end neural machine translation pipeline that translates English sentences into French using a series of recurrent neural network (RNN) architectures. The goal of the project is not to optimize translation quality, but to explore and compare different deep learning architectures commonly used in sequence-to-sequence learning tasks, while demonstrating a clear understanding of preprocessing, model design, training, and inference.

### Data Preprocessing

The preprocessing stage converts raw text into a numerical representation suitable for neural networks. Sentences are tokenized at the word level using Keras’ `Tokenizer`, mapping each unique word to an integer index. Word-level tokenization was chosen over character-level tokenization to reduce input complexity and improve learning efficiency, as word-level models generally converge faster for translation tasks.

Since neural networks require fixed-length inputs, all sequences are padded to a uniform length using post-padding. This ensures temporal alignment across batches while preserving the original word order. The preprocessing pipeline outputs padded English inputs, padded French labels reshaped to meet the requirements of sparse categorical cross-entropy, and the corresponding tokenizers for later decoding.

### Model Architectures

To explore different modeling strategies, five progressively more complex architectures were implemented:

1. **Simple RNN Model**  
   A baseline model using a Gated Recurrent Unit (GRU) followed by a TimeDistributed dense layer. This model establishes a reference point for sequence learning using numeric inputs without embeddings.

2. **Embedding RNN Model**  
   This model introduces a trainable word embedding layer, allowing the network to learn dense semantic representations of words rather than relying on raw integer indices. The embedding layer improves contextual understanding and leads to noticeably better performance compared to the baseline RNN.

3. **Bidirectional RNN Model**  
   A bidirectional GRU processes sequences in both forward and backward directions, allowing the model to incorporate both past and future context at each time step. This architecture improves sequence understanding and demonstrates how bidirectionality enhances translation tasks.

4. **Encoder–Decoder Model**  
   The encoder–decoder architecture compresses the input sentence into a fixed length context vector using an encoder GRU, which is then expanded by a decoder GRU to generate the translated sequence. This model reflects the foundational structure of many modern sequence to sequence systems.

5. **Final Combined Model (Encoder–Decoder with Attention)**
   The final model extends the encoder–decoder architecture by incorporating an attention mechanism in addition to word embeddings and a bidirectional encoder. Instead of relying on a single fixed-length context vector, the attention mechanism allows the decoder to dynamically focus on relevant encoder hidden states at each decoding step. This significantly improves alignment between input and output sequences and reduces common failure modes such as word repetition, resulting in more accurate and stable translations.

Each model was compiled with the Adam optimizer and trained using sparse categorical cross-entropy, which is appropriate for multi class sequence prediction with integer labels.

### Training and Prediction

All models were trained on the provided dataset and evaluated by generating predictions on training examples. Predictions were decoded back into human-readable French text using the learned token mappings. While the generated translations are not always linguistically perfect, they demonstrate that the models successfully learn word order, structure, and partial semantic alignment between English and French. In particular, the attention-based final model achieves substantially higher accuracy and produces more coherent translations by explicitly modeling alignment between source and target sequences.

The final prediction stage includes inference on both a custom English sentence and a dataset sentence, confirming that the full translation pipeline from raw text input to translated output is functional.

### Evaluation Considerations

For consistency with the project specification, models were trained on the full dataset without an explicit train/test split. While this leads to overstated accuracy values, the focus of the project is architectural understanding rather than generalization performance. A train/test split could be introduced in future work to provide a more realistic evaluation of model performance, but it was intentionally omitted to remain aligned with the assignment requirements.

### Conclusion

Overall, this project demonstrates a comprehensive understanding of neural machine translation workflows, including preprocessing, sequence modeling, architectural trade offs, and inference. By progressively building and comparing multiple RNN-based architectures, the project highlights how different design choices affect learning behavior and model expressiveness, fulfilling all functional and conceptual objectives of the assignment. While this project focuses on architectural exploration rather than strict generalization, the attention-based model demonstrates how appropriate architectural choices can significantly improve performance even without changing the dataset or training procedure.
